# Auto Evidence 360: Real-Source Quality Review

## tl;dr

- The current downloaded snapshot contains **1,411,783 real public-data rows** across seven analytical sources.
- All seven files matched their documented field counts; the current profile found no malformed rows.
- Exact normalized make/model/year matching is published as two measures: all valid keys (model years 1900 through current plus one) and EPA/NCAP-era-eligible keys (model year 1984 or later). Both are shown in the coverage table below; rates vary widely by source, proving that controlled entity resolution is a central project requirement.
- Public complaint PII-like fields and narratives are excluded from Fabric upload extracts.
- Complaint and bulletin volume are evidence signals, not make/model reliability rates.

## Context & Methods

The business question is simple: which make/model/year combinations deserve deeper review before a used vehicle is bought or listed?

### Key Assumptions

- A public record is evidence that an event or filing exists, not proof that every covered vehicle is defective.
- Cross-source comparisons require a transparent vehicle identity bridge.
- Counts without a make/model/year exposure denominator must not be labeled failure or reliability rates.
- The notebook emits aggregate checks only. It never displays complaint narratives, VIN fragments, contact fields, cities, or vehicle-operator fields.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config" / "sources.json").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "config" / "sources.json").exists(), "Project root could not be resolved"
subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "analysis" / "profile_real_sources.py")],
    cwd=PROJECT_ROOT,
    check=True,
)
profile_path = PROJECT_ROOT / "analysis" / "output" / "source_profile.json"
profile = json.loads(profile_path.read_text(encoding="utf-8"))
profile["generated_at_utc"]

## Data

Each source is downloaded from the publisher URL in `config/sources.json`. The downloader writes retrieval metadata and a SHA-256 checksum beside every file. Official NHTSA dictionaries define the tab-delimited schemas.

In [ ]:
dataset_profile = pd.DataFrame(profile["datasets"])
display_columns = [
    "source_id", "rows", "columns", "malformed_rows", "exact_duplicate_rows",
    "distinct_normalized_vehicle_keys", "min_year", "max_year", "min_source_date", "max_source_date"
]
dataset_profile[display_columns].fillna("n/a")

## Results

The first result checks whether the source files can be parsed at their documented shape. The second quantifies the entity-resolution gap before any alias or token matching is allowed.

In [ ]:
total_rows = int(dataset_profile["rows"].fillna(0).sum())
total_malformed = int(dataset_profile["malformed_rows"].fillna(0).sum())
pd.DataFrame({
    "metric": ["Downloaded analytical rows", "Malformed rows", "Sources profiled"],
    "value": [total_rows, total_malformed, int((dataset_profile["status"] == "profiled").sum())],
})

In [ ]:
match_coverage = pd.DataFrame(profile["cross_source_exact_match_coverage"])
match_coverage.assign(
    exact_match_rate=match_coverage["exact_match_rate"].map(lambda value: f"{value:.2%}"),
    era_eligible_exact_match_rate=match_coverage["era_eligible_exact_match_rate"].map(
        lambda value: f"{value:.2%}" if value is not None else "n/a"
    ),
)

The second coverage view shows the union of the four operational sources against the EPA/NCAP reference, again split into all-valid and era-eligible measures:

In [ ]:
union_coverage = profile["union_exact_match_coverage"]
pd.DataFrame([
    {"measure": "All valid vehicle keys", "keys": union_coverage["all_valid_vehicle_keys"],
     "exact matches": union_coverage["all_valid_exact_reference_matches"],
     "rate": f"{union_coverage['all_valid_exact_match_rate']:.2%}"},
    {"measure": "EPA/NCAP-era-eligible keys", "keys": union_coverage["era_eligible_vehicle_keys"],
     "exact matches": union_coverage["era_eligible_exact_reference_matches"],
     "rate": f"{union_coverage['era_eligible_exact_match_rate']:.2%}"},
])

## Takeaways

1. **The data are large enough for a credible Fabric project.** Complexity comes from multiple grains and schemas, not fabricated volume.
2. **Entity resolution is measurable work.** Exact matching is the high-confidence baseline; aliases and token rules need reviewed mappings and coverage reporting.
3. **Repeated business IDs can be legitimate.** A campaign or bulletin can repeat for multiple vehicles or components, so semantic measures must count the correct business entity.
4. **Privacy minimization happens before cloud upload.** Only approved fields in `data/fabric_upload/` enter Fabric.
5. **The dashboard must use precise language.** It reports public-record signals and review priority, not a universal safety or reliability ranking.